# Taxonomy Extraction

Run the enriched taxonomy prompt on papers predicted as POSITIVE by the classifier.
Uses the `taxonomy` task config from `llm_labeller.py`.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.metrics import cohen_kappa_score

sys.path.insert(0, str(Path().resolve().parent))

In [2]:
DATA_DIR = Path("../data")
SPLITS_DIR = DATA_DIR / "splits"

In [3]:
from src.labelling.llm_labeller import (
    claude_batch_submit,
    claude_batch_results,
    mistral_batch_submit,
    mistral_batch_results,
    google_batch_submit,
    google_batch_results,
    label_papers_async,
)

c:\Users\samma\Documents\Programmation\UdeM\nlp-benchmark-taxonomy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load positive predictions

In [4]:
predictions = pd.read_parquet(
    DATA_DIR / "predictions" / "inference_predictions_base.parquet"
)

# Keep only predicted positives
positives = predictions[predictions["predicted"] == 1].copy()
print(f"Positive predictions: {len(positives):,} papers")


Positive predictions: 3,738 papers


In [5]:
anthology = pd.read_parquet(DATA_DIR / "raw" / "anthology_enriched.parquet")

# Filter to positives in the anthology
positives_paper = anthology[anthology["bibkey"].isin(positives["bibkey"])]
print(f"Positive predictions in anthology: {len(positives_paper):,} papers")
# paper avant 2020
positives_paper = positives_paper[
    # (positives_paper["year"] < 2025) &
    (positives_paper["year"] >= 2025)
]
print(f"Positive predictions in anthology from 2025: {len(positives_paper):,} papers")


# en piocher max 300 au hasard
# positives_paper = positives_paper.sample(n=300, random_state=42)

Positive predictions in anthology: 3,738 papers
Positive predictions in anthology from 2025: 670 papers


## 2. Submit batch 

### Gemini

In [ ]:
job_id_gemma = google_batch_submit(
    positives_paper, task="taxonomy", model="gemini-3-flash-preview"
)
job_id_gemma


In [ ]:
id_2022_2025 = "batches/mlc3awi1fvond0kkyk1i1sowqrbsxzfk2fyy"
id_2017_2022 = "batches/11nmfv8wuh9jglety9d6pi766k2n2q56hueu"
id_2025 = "batches/wambcz3j9abkwwtyrcslkati4kq3jty1574i"

In [ ]:
result_gemma = google_batch_results(id_2025, positives_paper, task="taxonomy")
result_gemma.to_parquet(
    DATA_DIR / "labeled" / "google_taxonomy_2025.parquet", index=False
)


In [ ]:
results_gemma_2022_2025 = pd.read_parquet(
    DATA_DIR / "labeled" / "google_taxonomy_2022-2025.parquet"
)
results_gemma_2017_2022 = pd.read_parquet(
    DATA_DIR / "labeled" / "google_taxonomy_2017-2022.parquet"
)
results_gemma_2017 = pd.read_parquet(
    DATA_DIR / "labeled" / "google_taxonomy_results_4.parquet"
)

results_gemma_full = pd.concat(
    [results_gemma_2022_2025, results_gemma_2017_2022, results_gemma_2017],
    ignore_index=True,
)

results_gemma_full.to_parquet(
    DATA_DIR / "labeled" / "google_taxonomy_full.parquet", index=False
)

### Claude

In [ ]:
job_id = claude_batch_submit(positives_paper, task="taxonomy")
job_id

In [ ]:
results = claude_batch_results(job_id, positives_paper, task="taxonomy")
results.to_parquet(DATA_DIR / "labeled" / "claude_taxonomy_4.parquet", index=False)

### Deepseek

In [ ]:
# DeepSeek — async concurrent (no batch API available)
results_deepseek = await label_papers_async(
    positives_paper,
    provider="deepseek",
    task="taxonomy",
    max_concurrent=20,
    checkpoint_path=str(DATA_DIR / "labeled" / "deepseek_taxonomy_checkpoint.parquet"),
)

In [ ]:
results_deepseek.to_parquet(
    DATA_DIR / "labeled" / "deepseek_taxonomy_full.parquet", index=False
)

### Mistral

In [ ]:
mistral_job_id = mistral_batch_submit(positives_paper, task="taxonomy")
mistral_job_id

In [ ]:
results_mistral = mistral_batch_results(
    mistral_job_id, positives_paper, task="taxonomy"
)
results_mistral.to_parquet(
    DATA_DIR / "labeled" / "mistral_taxonomy_full.parquet", index=False
)

## 3. Analyse Predictions

In [4]:
results_deepseek = pd.read_parquet(
    DATA_DIR / "labeled" / "deepseek_taxonomy_full.parquet"
)
results_claude = pd.read_parquet(DATA_DIR / "labeled" / "claude_taxonomy_4.parquet")
results_mistral = pd.read_parquet(
    DATA_DIR / "labeled" / "mistral_taxonomy_full.parquet"
)
results_google = pd.read_parquet(DATA_DIR / "labeled" / "google_taxonomy_full.parquet")

In [10]:
from collections import Counter

LABEL_COLS = {
    "Google": "google_label",
    "DeepSeek": "deepseek_label",
    "Mistral": "mistral_label",
}

results_merged = (
    results_google[
        [
            "bibkey",
            "google_label",
            "google_benchmark_names",
            "google_task_type",
            "google_input_type",
            "google_label_type",
            "google_domain",
            "google_languages",
            "google_is_multitask",
        ]
    ]
    .merge(
        results_deepseek[
            [
                "bibkey",
                "deepseek_label",
                "deepseek_benchmark_names",
                "deepseek_task_type",
                "deepseek_input_type",
                "deepseek_label_type",
                "deepseek_domain",
                "deepseek_languages",
                "deepseek_is_multitask",
            ]
        ],
        on="bibkey",
    )
    .merge(
        results_mistral[
            [
                "bibkey",
                "mistral_label",
                "mistral_benchmark_names",
                "mistral_task_type",
                "mistral_input_type",
                "mistral_label_type",
                "mistral_domain",
                "mistral_languages",
                "mistral_is_multitask",
            ]
        ],
        on="bibkey",
    )
    .dropna(subset=list(LABEL_COLS.values()))
)


def _majority_and_dissenter(row):
    votes = {name: row[col] for name, col in LABEL_COLS.items()}
    counts = Counter(votes.values())
    winner, n = counts.most_common(1)[0]
    dissenters = [k for k, v in votes.items() if v != winner]
    return pd.Series(
        {
            "majority_label": winner,
            "dissenter": dissenters[0] if dissenters else None,
        }
    )


results_merged = pd.concat(
    [results_merged, results_merged.apply(_majority_and_dissenter, axis=1)],
    axis=1,
)

In [11]:
results_merged.to_parquet(DATA_DIR / "labeled" / "taxonomy_full.parquet", index=False)

### Cohens Kappa

In [ ]:
kappa = cohen_kappa_score(
    results_merged["google_label"],
    results_merged["deepseek_label"],
)
kappa_mistral = cohen_kappa_score(
    results_merged["deepseek_label"],
    results_merged["mistral_label"],
)

kappa_claude_mistral = cohen_kappa_score(
    results_merged["google_label"],
    results_merged["mistral_label"],
)

print(f"Cohen's kappa (Deepseek vs Google): {kappa:.3f}")
print(f"Cohen's kappa (Deepseek vs Mistral): {kappa_mistral:.3f}")
print(f"Cohen's kappa (Google vs Mistral): {kappa_claude_mistral:.3f}")